# 2장 3강: MSE와 Cross Entropy의 수학적 원리 — 실습문제

## 실습 목표

- 실제값에서 예측값을 빼고, 제곱한 뒤 평균을 내어 MSE를 구합니다.
- 실제 구매 여부에 맞는 확률을 사용해 BCE를 계산합니다.
- 실제 상품 종류에 해당하는 확률을 골라 Categorical CE를 계산합니다.
- 주어진 Softmax 함수로 점수를 확률로 바꾸고 CE를 구합니다.
- 교안에서 MSE·BCE·CE가 NLL과 연결되는 뜻을 쉬운 설명으로 확인합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 별도의 데이터 파일은 필요하지 않습니다. 데이터와 계산 방법은 제공된 「2장 3강: MSE와 Cross Entropy의 수학적 원리」의 본문 예제와 연습문제를 사용합니다.

회귀의 오차는 교안에 맞추어 **실제값 − 예측값**으로 계산합니다. 이진 분류는 **0=미구매, 1=구매**, 상품 분류는 **0=의류, 1=식품, 2=가전**입니다. 같은 숫자라도 문제에 따라 뜻이 다르므로 먼저 표의 제목을 확인하세요.

## 실습 구성과 교안 대응

| 구분 | 계산할 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | 두 금액 예측 모델의 오차·제곱오차·SSE·MSE | 1절 본문: 실제값 [10,20,30] 예제 |
| 필수 1 / 문제 1-2 | 고객별 BCE와 평균 BCE 비교 | 2절 본문: 고객 4명의 두 모델 비교 |
| 필수 2 / 문제 2-1 | 실제 정답 열 선택, 상품별 CE와 평균 CE | 3절 본문: 의류·식품·가전 상품 3개 예제 |
| 필수 2 / 문제 2-2 | 제공된 Softmax 함수로 확률과 CE 계산 | 3절 본문: Logit [2,1,0], [0,2,1], [1,0,3] |
| 과제 / 문제 3-1 | 새 금액 예측의 MSE 복습 | 연습문제 1의 MSE 계산 부분 |
| 과제 / 문제 3-2 | 새 고객 데이터의 BCE 복습 | 연습문제 2의 고객별·평균 BCE 계산 부분 |

**교안의 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제의 일부를 기초 실습 4문제와 복습 과제 2문제로 재구성했습니다. 과제에서도 앞에서 익힌 계산식과 교안에 있는 숫자를 사용합니다.

## 진행 안내

1. NumPy를 불러오는 셀부터 실행합니다.
2. 표를 읽고 실제값·예측값 또는 실제 정답·예측 확률을 구분합니다.
3. 제곱은 각 오차에 먼저 적용하고, 그다음 평균을 구합니다.
4. 분류에서는 **실제 정답에 부여한 확률**을 찾은 다음 음의 로그를 적용합니다.
5. 제공된 함수와 확률 선택 코드는 그대로 사용해도 됩니다. 공식을 유도하거나 함수를 외워서 작성할 필요는 없습니다.
6. 계산 중간값은 반올림하지 않고 출력에만 `round()`·`np.round()`를 사용합니다.
7. 각 문제의 Q1~Q4에 계산값과 쉬운 문장으로 답하세요.

## 먼저 읽는 용어와 수식 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 손실 함수 / Loss Function | 모델의 예측을 정답과 비교하여 손실값으로 계산하는 규칙 |
| 회귀 / 분류 | 금액처럼 수치를 예측 / 정해진 종류를 예측 |
| y, y_true | 실제 정답. 금액인지 구매 여부인지 상품 종류인지 문제에 따라 확인 |
| ŷ, y_pred | 모델이 예측한 수치. ŷ는 ‘와이 햇’으로 읽음 |
| 오차 / errors | 실제값에서 예측값을 뺀 수 |
| 제곱 / ** 2 | 같은 수를 두 번 곱함. (−2)²=(−2)×(−2)=4 |
| SSE | 제곱오차를 모두 더한 값 |
| MSE | 제곱오차의 평균. SSE를 데이터 개수로 나눈 값 |
| p / 1−p | 이진 분류의 구매 확률 / 미구매 확률 |
| 클래스 / 인덱스 | 종류의 이름 / 배열에서의 위치 번호. 첫 위치는 0 |
| BCE | Binary Cross Entropy. 이 자료의 구매·미구매 분류에서 쓰는 손실 |
| CE | 이 자료에서는 Categorical Cross Entropy. 상품이 한 종류에만 속하는 분류의 손실 |
| One-Hot | 정답 클래스의 위치만 1, 나머지 위치는 0으로 표시한 배열 |
| Logit / Softmax | 확률로 바꾸기 전 점수 / 여러 점수를 합이 1인 확률로 바꾸는 함수 |
| n / Σ / ≈ | 데이터 개수 / 여러 값을 더하라는 기호 / 대략 같음 |
| ln / np.log | 자연로그 / 자연로그를 계산하는 NumPy 코드 |
| −ln(p) | 먼저 p의 자연로그를 구한 뒤 부호를 바꿈 |
| NLL | Negative Log-Likelihood. 우도의 로그에 마이너스를 붙인 값 |
| Mean / Total | 평균 / 합계 |

**로그를 처음 보는 경우:** 로그는 ‘어떤 수를 몇 제곱하면 주어진 값이 되는가’를 묻는 계산입니다. 자연로그는 약 2.718인 수 e를 기준으로 합니다. 이번 실습에서는 로그값을 손으로 구하지 않고 `np.log()`를 사용합니다. 예를 들어 `np.log(0.8)`은 약 −0.2231이고, `-np.log(0.8)`은 부호를 바꾼 약 0.2231입니다. 0과 1 사이 확률에 로그를 적용하면 음수이므로 앞의 마이너스를 빠뜨리지 마세요.

**손실 비교의 기준:** 같은 데이터·같은 손실 함수에서 값이 작은 쪽을 선택합니다. MSE와 BCE처럼 서로 다른 손실값을 숫자 크기만으로 비교하지 않습니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 배열을 계산하는 도구이고, `np`는 NumPy를 짧게 부르는 이름입니다.

In [1]:
import numpy as np

---

## 필수 1. 회귀와 이진 분류의 손실 계산하기

### 문제 1-1. 오차를 제곱하고 평균 내어 MSE 비교하기

#### 문제 설명

교안 1절 본문의 구매 금액 예측입니다. 단위는 **만 원**입니다.

| 고객 | 실제값 | 모델 A 예측 | 모델 B 예측 |
|---|---:|---:|---:|
| 첫째 | 10 | 12 | 9 |
| 둘째 | 20 | 18 | 24 |
| 셋째 | 30 | 27 | 33 |

오차는 실제값−예측값, 제곱오차는 오차×오차입니다. SSE는 제곱오차의 합, MSE는 제곱오차의 평균입니다.

#### 요구사항

1. 실제값과 두 모델의 예측값을 각각 배열로 만드세요.
2. 두 모델의 오차와 제곱오차를 계산해 출력하세요.
3. `np.sum()`으로 SSE, `np.mean()`으로 MSE를 구하세요. MSE는 소수점 아래 세 자리로 표시합니다.
4. Q1~Q4에 답하세요.

#### 작성 도움말

```python
errors = y_true - y_pred
squared_errors = errors ** 2
sse = np.sum(squared_errors)
mse = np.mean(squared_errors)
```

먼저 각 오차를 제곱해야 합니다. 오차를 먼저 평균한 뒤 제곱하는 계산과 다릅니다. 위의 변수 이름을 두 모델에 맞게 구분해서 사용하세요.

#### 해석 질문

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

**Q2.** SSE와 MSE는 각각 얼마이며, 어느 모델을 선택하나요?

**Q3.** 모델 A의 첫 오차 −2는 실제보다 높게 예측했다는 뜻인가요, 낮게 예측했다는 뜻인가요?

**Q4.** MSE 5.667을 평균 5.667만 원 틀렸다고 해석해도 되나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [2]:
# 문제 1-1. 오차를 제곱하고 평균 내어 MSE 비교하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1
y_true = np.array([10, 20, 30])

y_pred_a = np.array([12, 18, 27])
y_pred_b = np.array([9, 24, 33])

# 2
# 모델 A의 오차 구하기
# 첫 번째는 10-12 = -2 값을 의미 
errors_a = y_true - y_pred_a
errors_b = y_true - y_pred_b

squared_a = errors_a ** 2
squared_b = errors_b ** 2

# 3
# 합계와 평균 구하기
sse_a = np.sum(squared_a)
sse_b = np.sum(squared_b)

mse_a = np.mean(squared_a)
mse_b = np.mean(squared_b)

print('모델 A 오차:', errors_a)
print('모델 B 오차:', errors_b)
print('모델 A 제곱오차:', squared_a)
print('모델 B 제곱오차:', squared_b)
print('모델 A SSE:', sse_a)
print('모델 B SSE:', sse_b)
print('모델 A MSE:', np.round(mse_b, 3))
print('모델 B MSE:', np.round(mse_b, 3))

모델 A 오차: [-2  2  3]
모델 B 오차: [ 1 -4 -3]
모델 A 제곱오차: [4 4 9]
모델 B 제곱오차: [ 1 16  9]
모델 A SSE: 17
모델 B SSE: 26
모델 A MSE: 8.667
모델 B MSE: 8.667


#### 답변 작성란

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

답변: 
- A는 오차가 [-2, 2, 3], 제곱오차 [4, 4, 9]
- B는 오차가 [1, -4, -3], 제곱오차 [1, 16, 9]

**Q2.** SSE와 MSE는 각각 얼마이며, 어느 모델을 선택하나요?

답변: 
- A의 SSE는 17, MSE는 17 / 3 = 5.66
- B의 SSE는 26, MSE는 26 / 3 = 8.66
- -> 동일한 데이터의 MSE가 더 작은 A모델을 선택하는 것이 합리적

**Q3.** 모델 A의 첫 오차 −2는 실제보다 높게 예측했다는 뜻인가요, 낮게 예측했다는 뜻인가요?

답변: 
- 높게 예측
- 실제는 10만원인데, 예측은 12만원이니까 실제보다 2만원 높음(10 - 12 = -2)


**Q4.** MSE 5.667을 평균 5.667만 원 틀렸다고 해석해도 되나요?

답변:
- 아니요,
- MSE는 오차 자체의 평균이 아니라 제곱오차의 평균
- -> 만원을 제곱한 (만원)² 단위로 해석해야함

### 문제 1-2. 실제 구매 여부에 맞추어 BCE 계산하기

#### 문제 설명

교안 2절 본문의 두 모델입니다. 실제 정답은 1=구매, 0=미구매이며 p는 항상 구매 확률입니다.

| 고객 | 실제 정답 y | 모델 A의 p | 모델 B의 p |
|---|---:|---:|---:|
| 첫째 | 1 | 0.9 | 0.9 |
| 둘째 | 0 | 0.2 | 0.2 |
| 셋째 | 1 | 0.6 | 0.8 |
| 넷째 | 0 | 0.7 | 0.1 |

고객 한 명의 BCE는 `−[y×ln(p)+(1−y)×ln(1−p)]`입니다. y=1이면 −ln(p), y=0이면 −ln(1−p)로 계산합니다.

#### 요구사항

1. 실제 정답과 모델별 구매 확률을 배열로 만드세요.
2. 아래 함수를 그대로 사용하여 두 모델의 고객별 BCE를 구하세요.
3. 각 손실 배열에 `np.mean()`을 적용해 평균 BCE를 구하세요. 고객별·평균 손실을 소수점 아래 세 자리로 출력합니다.
4. Q1~Q4에 답하세요.

#### 제공 함수와 작성 도움말

```python
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))
```

이 함수는 고객별 손실 배열을 반환합니다. 평균은 함수 밖에서 구합니다. 예를 들어 실제 정답이 0이고 구매 확률이 0.7이면, 실제 정답인 미구매의 확률 1−0.7=0.3을 사용합니다. 제공된 확률은 모두 0보다 크고 1보다 작아 이 식으로 계산할 수 있습니다.

#### 해석 질문

**Q1.** 모델 A가 실제 정답에 부여한 확률은 각각 얼마인가요?

**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

**Q3.** 미구매한 넷째 고객의 모델 A 손실에 왜 0.7이 아니라 0.3을 사용하나요?

**Q4.** 어느 모델을 선택하며, 평균 손실이 작아진 이유는 무엇인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [3]:
# 문제 1-2. 실제 구매 여부에 맞추어 BCE 계산하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1
# 실제 결과와 구매 확률을 맞추기
# 마지막 고객은 실제로 0이지만 A는 구매확률을 0.7로 주었다는 것이 특징
y_true = np.array([1, 0, 1, 0])

p_a = np.array([0.9, 0.2, 0.6, 0.7])
p_b = np.array([0.9, 0.2, 0.8, 0.1])

# 2
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))

loss_a = binary_cross_entropy(y_true, p_a)
loss_b = binary_cross_entropy(y_true, p_b)

# 3
mean_a = np.mean(loss_a)
mean_b = np.mean(loss_b)

print('모델 A 고객별 BCE :', np.round(loss_a, 3))
print('모델 B 고객별 BCE :', np.round(loss_b, 3))
print('모델 A 평균 BCE :', np.round(mean_a, 3))
print('모델 B 평균 BCE :', np.round(mean_b, 3))

모델 A 고객별 BCE : [0.105 0.223 0.511 1.204]
모델 B 고객별 BCE : [0.105 0.223 0.223 0.105]
모델 A 평균 BCE : 0.511
모델 B 평균 BCE : 0.164


#### 답변 작성란

**Q1.** 모델 A가 실제 정답에 부여한 확률은 각각 얼마인가요?

답변: 
- 고객 순서대로 [0.9, 0.2, 0.6, 0.7]이다. 실제값이 0인 둘째와 넷째에는 구매확률  p 대신 1-p를 사용함

**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

답변: 
- A의 고객별 BCE는 약 [0.105 0.223 0.511 1.204] 평균 약 0.511
- B의 고객별 BCE는 약 [0.105 0.223 0.223 0.105] 평균 약 0.164


**Q3.** 미구매한 넷째 고객의 모델 A 손실에 왜 0.7이 아니라 0.3을 사용하나요?

답변: 
- 0.7은 구매확률이고 실제 정답은 미구매였기 때문이다. 미구매 확률은 1-0.7 = 0.3
- -> 0.7을 쓰게 되면 손실을 구매가 정답인 경우의 손실을 구하기 때문에 0.3을 쓰는 것이 더 적절하다.

**Q4.** 어느 모델을 선택하며, 평균 손실이 작아진 이유는 무엇인가요?

답변:
- 평균 BCE가 약 0.164로 더 작은 B를 선택한다.
- B는 셋째 실제 구매 확률을 0.6에서 0.8로, 넷째의 실제 미구매 확률을 0.3에서 0.9로 높임
- -> 실제 정답에 더 높은 확률을 주어서 손실을 줄였음

---

## 필수 2. 다중 클래스 분류의 손실 계산하기

### 문제 2-1. 실제 정답 클래스의 확률로 CE 계산하기

#### 문제 설명

교안 3절의 상품 3개 예제입니다. 클래스 순서는 **의류 0, 식품 1, 가전 2**입니다. 한 상품의 실제 정답은 한 종류입니다.

| 상품 | 실제 정답 | 의류 확률 | 식품 확률 | 가전 확률 |
|---|---|---:|---:|---:|
| A | 의류 | 0.8 | 0.1 | 0.1 |
| B | 식품 | 0.2 | 0.6 | 0.2 |
| C | 가전 | 0.1 | 0.3 | 0.6 |

상품 하나의 `CE = −ln(실제 정답 클래스의 확률)`입니다. 평균 CE는 상품별 CE를 더하고 상품 수로 나눕니다.

#### 요구사항

1. 정답 배열 `y_true = np.array([0,1,2])`와 표의 3행 3열 확률 배열 `y_prob`를 만드세요.
2. 아래 교안의 코드로 상품별 실제 정답 확률을 골라 출력하세요.
3. `-np.log(true_class_prob)`로 상품별 CE를 구하고 `np.mean()`으로 평균을 구하세요. 손실은 소수점 아래 세 자리로 출력합니다.
4. Q1~Q4에 답하세요.

#### 제공 코드와 읽는 방법

```python
rows = np.arange(len(y_true))
true_class_prob = y_prob[rows, y_true]
```

rows는 상품의 행 번호 [0,1,2]입니다. y_true는 그 상품의 정답 열 번호입니다. 두 배열의 같은 위치를 짝지어 (0행,0열), (1행,1열), (2행,2열)을 고릅니다. 인덱스는 0부터 셉니다. 코드는 그대로 사용해도 됩니다.

#### 해석 질문

**Q1.** 각 상품에서 어느 행·열을 선택하며, 선택한 확률은 얼마인가요?

**Q2.** 상품별 CE와 평균 CE는 각각 얼마인가요?

**Q3.** 평균 CE에서 3으로 나누는 이유는 무엇인가요?

**Q4.** CE에 사용할 확률은 가장 큰 예측 확률인가요, 실제 정답 클래스의 확률인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [4]:
# 문제 2-1. 실제 정답 클래스의 확률로 CE 계산하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 행과 열을 구분
# 가로줄인 행은 식품 A / B / C를 의미
# 세로 줄인 열은 의류 / 식품 / 가전을 의미
# 첫 행 [0.8, 0.1, 0.1]은 한 상품 A의 세 가지 클래스 확률을 의미

# 1
y_true = np.array([0,1,2])

y_prob = np.array([
    [0.8, 0.1, 0.1],
    [0.2, 0.6, 0.2],
    [0.1, 0.3, 0.6]
])

# 2
rows = np.arange(len(y_true))
true_class_prob = y_prob[rows, y_true]

# 3
# 정답 확률에 음의 로그를 적용
ce_each = -np.log(true_class_prob)
mean_ce = np.mean(ce_each)

print('상품의 행 번호 :', rows)
print('실제 정답 클래스 확률 :', true_class_prob) 
print('상품별 CE :', np.round(ce_each, 3))
print('평균 CE :', np.round(mean_ce, 3))

상품의 행 번호 : [0 1 2]
실제 정답 클래스 확률 : [0.8 0.6 0.6]
상품별 CE : [0.223 0.511 0.511]
평균 CE : 0.415


#### 답변 작성란

**Q1.** 각 상품에서 어느 행·열을 선택하며, 선택한 확률은 얼마인가요?

답변: 
- 상품 A는 0행 0열의 0.8
- B는 1행 1열의 0.6
- C는 2행 2열의 0.6, 열은 그 상품의 실제 정답 클래스

**Q2.** 상품별 CE와 평균 CE는 각각 얼마인가요?

답변: 
- 상품별로 -ln(0.8), -ln(0.6), -ln(0.6)이니까 약 [0.223 0.511 0.511]
- 반올림 전 세 값을 더해 3으로 나누면 평균 CE는 약 0.415

**Q3.** 평균 CE에서 3으로 나누는 이유는 무엇인가요?

답변: 
- 손실은 계산하려는 상품이 3개여서 상품별 CE 세 개의 평균을 구하는 계산이기 때문

**Q4.** CE에 사용할 확률은 가장 큰 예측 확률인가요, 실제 정답 클래스의 확률인가요?

답변:
- 실제 정답 클래스의 확률, 가장 큰 예측 확률은 모델이 어느 클래스를 선택했는지 알려준다.
- 그 클래스가 실제 정답과 다를 수 있기 때문에 CE에서는 y_true가 가르키는 정답 열을 사용한다.

### 문제 2-2. 주어진 Softmax 함수로 확률과 CE 계산하기

#### 문제 설명

교안 3절의 점수(Logit) 예제입니다. 클래스 순서는 의류 0, 식품 1, 가전 2입니다.

| 상품 | 의류 점수 | 식품 점수 | 가전 점수 | 실제 정답 인덱스 |
|---|---:|---:|---:|---:|
| A | 2 | 1 | 0 | 0 |
| B | 0 | 2 | 1 | 1 |
| C | 1 | 0 | 3 | 2 |

점수는 아직 확률이 아닙니다. 아래 **교안의 Softmax 함수를 그대로 사용**하여 확률로 바꾸세요.

```python
def softmax(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)
```

#### 요구사항

1. 점수 행렬 logits와 정답 배열 y_true를 만들고, 제공된 함수를 작성하세요.
2. `probabilities = softmax(logits)`로 확률을 구하고, `np.sum(probabilities, axis=1)`로 상품별 확률 합을 확인하세요.
3. `probabilities[np.arange(len(y_true)), y_true]`로 실제 정답 확률을 선택하세요. 상품별 CE와 평균 CE를 구하세요.
4. 확률과 손실은 소수점 아래 세 자리로 출력하고 Q1~Q4에 답하세요.

#### 작성 도움말

`axis=1`은 각 상품 행 안의 클래스 방향으로 계산하라는 뜻입니다. 정답 확률을 선택한 다음의 CE 계산은 문제 2-1과 같습니다. 손실 계산에는 반올림 전 확률을 사용하세요.

#### 해석 질문

**Q1.** Softmax 결과와 상품별 확률 합은 무엇인가요?

**Q2.** 실제 정답 확률, 상품별 CE, 평균 CE는 각각 얼마인가요?

**Q3.** 세 상품 모두 정답을 가장 높게 예측했는데 C의 손실이 더 작은 이유는 무엇인가요?

**Q4.** 점수 [2,1,0]을 곧바로 CE의 확률 자리에 넣어도 되나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [5]:
# 문제 2-2. 주어진 Softmax 함수로 확률과 CE 계산하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

def softmax(z):
    shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

logits = np.array([ # Logits, 2, 1, 0은 상품 A에 대한 각 클래스 점수
    [2, 1, 0],
    [0, 2, 1],
    [1, 0, 3]
])

y_true = np.array([0, 1, 2])
probabilities = softmax(logits)
row_sums = np.sum(probabilities, axis=1)
true_class_prob = probabilities[np.arange(len(y_true)), y_true]
losses = -np.log(true_class_prob)
mean_ce = np.mean(losses)

print('Softmax 확률:')
print(np.round(probabilities, 3))

print('상품별 확률 합:', np.round(row_sums, 3))
print('실제 정답 클래스 확률 :', np.round(true_class_prob, 3))

Softmax 확률:
[[0.665 0.245 0.09 ]
 [0.09  0.665 0.245]
 [0.114 0.042 0.844]]
상품별 확률 합: [1. 1. 1.]
실제 정답 클래스 확률 : [0.665 0.665 0.844]


#### 답변 작성란

**Q1.** Softmax 결과와 상품별 확률 합은 무엇인가요?

답변: 
- 확률은 [[0.665, 0.245, 0.090]], [[0.090, 0.065, 0.245]], [[0.114, 0.042, 0.844]]
- 각 상품 클래스의 확률을 더하면 1, 행렬 합은 [1, 1, 1]

**Q2.** 실제 정답 확률, 상품별 CE, 평균 CE는 각각 얼마인가요?

답변: 
- 정답 확률은 약 [0.665, 0.665, 0.844] 정도, 반올림 전 확률에 음의 로그를 적용하면
- CE는 약 [0.408, 0.408, 0.170] 평균은 약 0.328이다.

**Q3.** 세 상품 모두 정답을 가장 높게 예측했는데 C의 손실이 더 작은 이유는 무엇인가요?

답변: 
-  C가 실제 정답인 가전에 부여한 확률이 약 0.844로 높다.
- A와 B의 정답 확률은 약 0.665
- -> CE는 맞고 틀림이 아니라 실제 정답 확률을 반영하기 때문에 C의 손실이 가장 작다

**Q4.** 점수 [2,1,0]을 곧바로 CE의 확률 자리에 넣어도 되나요?

답변:
- 아니다
- [2, 1, 0]은 아직 확률이 아닌 점수
- 먼저 제공된 SoftMax로 확률을 구하고
- 실제 정답 클래스의 확률을 선택하여 음의 로그를 적용하는 순서

---

## 과제. 필수 실습과 같은 방법으로 복습하기

### 문제 3-1. 새로운 금액 예측의 MSE 복습하기

#### 문제 설명

교안 연습문제 1의 MSE 계산을 복습합니다. 금액 단위는 만 원입니다.

| 고객 | 실제값 | 모델 A 예측 | 모델 B 예측 |
|---|---:|---:|---:|
| 첫째 | 8 | 10 | 8 |
| 둘째 | 12 | 11 | 16 |
| 셋째 | 20 | 18 | 17 |

문제 1-1과 같은 순서로 계산하세요. 새 공식은 필요하지 않습니다.

#### 요구사항

1. 실제값과 두 예측 배열을 만드세요.
2. 각 모델의 오차와 제곱오차를 계산해 출력하세요.
3. SSE와 MSE를 구하고 MSE는 소수점 아래 세 자리로 출력하세요.
4. Q1~Q4에 답하세요.

#### 작성 도움말

`오차 = 실제값 − 예측값`이며, `MSE = np.mean(오차 ** 2)`입니다. 모델 A의 첫 오차는 8−10입니다. 평균의 분모는 고객 수 3입니다.

#### 해석 질문

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

**Q2.** SSE와 MSE는 각각 얼마이며 어떤 모델을 선택하나요?

**Q3.** 모델 B는 첫 고객을 정확히 예측했는데도 전체 MSE가 더 큰 이유는 무엇인가요?

**Q4.** 모델 B의 MSE를 계산할 때 오차가 있는 두 고객만 세어 2로 나누어도 되나요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [6]:
# 문제 3-1. 새로운 금액 예측의 MSE 복습하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1
y_true = np.array([8, 12, 20])

y_pred_a = np.array([10, 11, 18])
y_pred_b = np.array([8, 16, 17])

# 2
# 각 모델의 오차와 제곱오차
errors_a = y_true - y_pred_a
errors_b = y_true - y_pred_b

squared_a = errors_a ** 2
squared_b = errors_b ** 2

print('모델 A의 오차 :', errors_a)
print('모델 B의 오차 :', errors_b)
print('모델 A의 제곱오차 :', squared_a)
print('모델 B의 제곱오차 :', squared_b)

# 3
# SSE, MSE
sse_a = np.sum(squared_a)
sse_b = np.sum(squared_b)

mse_a = np.mean(squared_a)
mse_b = np.mean(squared_b)

print('모델 A SSE:', sse_a)
print('모델 B SSE:', sse_b)
print('모델 A MSE:', np.round(mse_a, 3))
print('모델 B MSE:', np.round(mse_b, 3))

모델 A의 오차 : [-2  1  2]
모델 B의 오차 : [ 0 -4  3]
모델 A의 제곱오차 : [4 1 4]
모델 B의 제곱오차 : [ 0 16  9]
모델 A SSE: 9
모델 B SSE: 25
모델 A MSE: 3.0
모델 B MSE: 8.333


#### 답변 작성란

**Q1.** 두 모델의 오차와 제곱오차는 각각 무엇인가요?

답변: 
- 모델 A의 오차는 [-2, 1, 2]
- 모델 B의 오차는 [ 0, -4 , 3]
- 모델 A의 제곱오차는 [4, 1, 4]
- 모델 B의 제곱오차는 [ 0, 16 , 9] 

**Q2.** SSE와 MSE는 각각 얼마이며 어떤 모델을 선택하나요?

답변: 
- 모델 A의 SSE는 9, 모델 B의 SSE는 25이다.
- 모델 A의 MSE는 3, 모델 B의 MSE는 8.333이다.
- 모델 A의 SSE와 MSE는 B보다 작으므로 모델 A를 선택한다. 

**Q3.** 모델 B는 첫 고객을 정확히 예측했는데도 전체 MSE가 더 큰 이유는 무엇인가요?

답변: 
- 모델 B는 첫 번째 고객의 오차가 0이지만, 두 번째와 세 번째 고객의 오차가 각각 -4, 3으로 A보다 크다.
- MSE는 제곱으로 계산하기 때문에 큰 오차에 더 큰 영향을 받는다. 
- 따라서 모델 B의 전체 MSE가 모델 A보다 크다.

**Q4.** 모델 B의 MSE를 계산할 때 오차가 있는 두 고객만 세어 2로 나누어도 되나요?

답변:
- 안 된다.
- 전체 데이터 개수로 나누어야 한다. 
- 따라서 모델 B의 MSE는 전체 고객 수인 3으로 나누어 계산해야 한다.

### 문제 3-2. 새로운 고객 데이터의 BCE 복습하기

#### 문제 설명

교안 연습문제 2의 고객 3명입니다. 실제값은 1=구매, 0=미구매입니다. 표의 확률은 구매 확률입니다.

| 고객 | 실제값 y | 모델 A 구매 확률 | 모델 B 구매 확률 |
|---|---:|---:|---:|
| 첫째 | 1 | 0.8 | 0.6 |
| 둘째 | 0 | 0.3 | 0.6 |
| 셋째 | 1 | 0.9 | 0.7 |

문제 1-2에서 사용한 BCE 함수를 그대로 사용합니다.

#### 요구사항

1. 실제값과 두 모델의 확률 배열을 만드세요.
2. 아래 함수를 이용해 각 모델의 고객별 BCE를 계산하세요.
3. `np.mean()`으로 두 평균 BCE를 구하세요. 고객별·평균 BCE는 소수점 아래 세 자리로 출력합니다.
4. Q1~Q4에 답하세요.

#### 제공 함수와 작성 도움말

```python
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))
```

실제 정답이 0인 둘째 고객은 미구매 확률을 사용합니다. 모델 A에서는 1−0.3, 모델 B에서는 1−0.6입니다. 평균의 분모는 고객 수 3입니다.

#### 해석 질문

**Q1.** 두 모델이 실제 정답에 부여한 확률은 각각 무엇인가요?

**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

**Q3.** 모델 B가 첫째와 둘째에게 같은 구매 확률 0.6을 주었는데도 BCE가 다른 이유는 무엇인가요?

**Q4.** 평균 BCE 기준으로 어느 모델을 선택하며, 그 이유는 무엇인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [7]:
# 문제 3-2. 새로운 고객 데이터의 BCE 복습하기
# 1. 문제에 제시된 배열과 필요한 함수를 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 결과와 그 의미를 적으세요.

# 1
y_true = np.array([1, 0, 1])

y_prob_a = np.array([0.8, 0.3, 0.9])
y_prob_b = np.array([0.6, 0.6, 0.7])


# 2
def binary_cross_entropy(y_true, y_prob):
    return -(y_true * np.log(y_prob)
             + (1 - y_true) * np.log(1 - y_prob))

# 고객별 BCE 계산
bce_a = binary_cross_entropy(y_true, y_prob_a)
bce_b = binary_cross_entropy(y_true, y_prob_b)

# 3. 평균 BCE 계산
mean_bce_a = np.mean(bce_a)
mean_bce_b = np.mean(bce_b)

print("모델 A 고객별 BCE:", np.round(bce_a, 3))
print("모델 B 고객별 BCE:", np.round(bce_b, 3))

print("모델 A 평균 BCE:", np.round(mean_bce_a, 3))
print("모델 B 평균 BCE:", np.round(mean_bce_b, 3))

모델 A 고객별 BCE: [0.223 0.357 0.105]
모델 B 고객별 BCE: [0.511 0.916 0.357]
모델 A 평균 BCE: 0.228
모델 B 평균 BCE: 0.595


#### 답변 작성란

**Q1.** 두 모델이 실제 정답에 부여한 확률은 각각 무엇인가요?

답변: 

- 모델 A가 실제 정답에 부여한 확률은 [0.8, 0.7, 0.9]이고, 
- 모델 B는 [0.6, 0.4, 0.7]이다.


**Q2.** 두 모델의 고객별 BCE와 평균 BCE는 각각 얼마인가요?

답변: 
- 모델 A 고객별 실제 정답에 부여한 확률은 [0.223, 0.357, 0.105]이고,
- 모델 B 고객별 실제 정답에 부여한 확률은 [0.511, 0.916, 0.357]이다.

- 모델 A 평균 BCE는 0.228이고, 
- 모델 B 평균 BCE는 0.595이다.

**Q3.** 모델 B가 첫째와 둘째에게 같은 구매 확률 0.6을 주었는데도 BCE가 다른 이유는 무엇인가요?

답변: 
- 첫째 고객은 실제로 구매했으므로 구매 확률 0.6을 기준으로 BCE를 계산하지만, 
- 둘째 고객은 실제로 구매하지 않았으므로 미구매 확률 1 - 0.6 = 0.4를 기준으로 계산한다. 


**Q4.** 평균 BCE 기준으로 어느 모델을 선택하며, 그 이유는 무엇인가요?

답변:
- BCE는 실제 정답에 높은 확률을 부여할수록 손실이 감소한다.
- 모델 A가 모델 B보다  BCE가 낮기 때문에 더 정확한 예측을 했다고 볼 수 있다.
- 따라서 모델 A를 선택한다.

---

## 실습 마무리

다음 질문으로 복습하세요. 6번은 교안의 연결 원리를 읽고 확인하는 질문입니다.

**1.** MSE를 구하는 계산 순서는 무엇인가요?

**2.** 실제값 10, 예측값 12일 때 오차 −2의 뜻은 무엇인가요?

**3.** 실제 미구매인데 구매 확률이 0.7이면 BCE에 어떤 확률을 쓰나요?

**4.** 상품 CE에서는 어떤 확률을 고르나요?

**5.** 상품별 CE를 구한 다음 평균은 무엇으로 나누나요?

**6.** MSE와 Gaussian NLL, BCE·CE와 NLL은 각각 어떻게 연결되나요?

### 실습 마무리 답변 작성란

**1.** 답변: 
- 1. 오차 = 실제값 - 예측값
- 2. 각 오차를 제곱 
- 3. 제곱오차를 모두 더함
- 4. 제곱 오차를 모두 더한 결과의 데이터 수로 나눔
- -> 제곱오차의 합을 SSE라고 부르고 , 평균은 MSE라고 부른다.

**2.** 답변: 
- 실제보다 2만큼 높게 예측했다는 뜻
- 오차는 10 - 12 = -2고, 제곱오차는 (-2)² = 4

**3.** 답변: 
- 미구매 확률 1 - 0.7 = 0.3을 사용한다.
- BCE는 -ln(0.3) = 약 1.20

**4.** 답변: 
- 각 상품의 실제 정답 클래스에 해당하는 확률을 고른다.
- 모델이 가장 높게 예측한 클래스가 실제 정답과 다르더라도 기준은 실제 정답

**5.** 답변: 
- 상품별 CE를 모두 더한 뒤 상품 수로 나눈다. (예제에서 분모가 3)

**6.** 답변:
- 독립적인 정규분포 오차와 동일한 고정 분산을 가정하여 평균 Gaussian NLL은 같은 상수에 양의 상수 x MSE를 더한 상태
- 값 자체는 달라도 같은 최적 평균 파라미터를 선택
- 한 데이터의 BCE, CE는 각각 베르누이 범주 분포의 NLL
- 조건부 독립 가정에서 데이터별 손실을 더하면 전체 NLL이고, 데이터 수로 나누면 BCE, CE이다.